In [3]:
import scanpy as sc
import squidpy as sq
import cell2location
import scvi
import torch
import pandas as pd
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.stats.multitest import multipletests
from pathlib import Path
import scipy.sparse
import logging
import warnings
from tqdm import tqdm
from scipy.stats import pearsonr
import libpysal
from esda.moran import Moran
from skspatial.objects import Points, Plane
from sklearn.preprocessing import StandardScaler
import esda 


# Set up logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    datefmt='%Y-%m-%d %H:%M:%S'
)
logger = logging.getLogger(__name__)

# Configure warnings
warnings.filterwarnings('ignore', category=RuntimeWarning)
warnings.filterwarnings('ignore', category=UserWarning)

class SpatialLRAnalysis:
    def __init__(self, visium_path_infected, visium_path_uninfected, sc_adata,
                new_annotations_adata, lr_interactions_path, output_dir="./results4"):
        """Initialize spatial L-R interaction analysis"""
        logger.info("Initializing analysis...")
    
        # Initialize cache
        self.score_cache = {}
        self.hotspot_cache = {}
    
        # Load Visium data
        logger.info("Loading Visium data...")
        try:
            self.visium_infected = sc.read_visium(visium_path_infected)
            self.visium_uninfected = sc.read_visium(visium_path_uninfected)
        except Exception as e:
            logger.error(f"Failed to load Visium data: {str(e)}")
            raise
    
        # Store single-cell data and update annotations
        self.sc_adata = sc_adata.copy()
        logger.info("Updating cell type annotations...")
        try:
            # Update cell type annotations while preserving original data structure
            self.sc_adata.obs['celltypes'] = new_annotations_adata.obs['celltypes_redo'][self.sc_adata.obs.index]
            # Update colors if they exist
            if 'celltypes_redo_colors' in new_annotations_adata.uns:
                self.sc_adata.uns['celltypes_colors'] = new_annotations_adata.uns['celltypes_redo_colors']
        except Exception as e:
            logger.error(f"Failed to update cell type annotations: {str(e)}")
            raise
    
        # Load L-R interactions
        logger.info("Loading L-R interactions...")
        try:
            self.lr_interactions = pd.read_csv(lr_interactions_path)
        except Exception as e:
            logger.error(f"Failed to load L-R interactions: {str(e)}")
            raise
    
        # Setup output directory
        self.output_dir = Path(output_dir)
        self.output_dir.mkdir(parents=True, exist_ok=True)
    
        # Initialize results storage
        self.spatial_patterns_infected = None
        self.spatial_patterns_uninfected = None
        self.differential_patterns = None
    
        # Setup data
        self._setup_data()

    def _setup_data(self):
        """Prepare and validate data for analysis"""
        logger.info("Setting up data...")
    
        try:
            # Validate cell type annotations
            if self.sc_adata.obs['celltypes'].isna().any():
                raise ValueError("Missing cell type annotations detected")
        
            # Get unique cell types for logging
            unique_celltypes = self.sc_adata.obs['celltypes'].unique()
            logger.info(f"Found {len(unique_celltypes)} unique cell types: {', '.join(unique_celltypes)}")
        
            # Verify spatial coordinates
            for adata, label in [(self.visium_infected, "infected"),
                                (self.visium_uninfected, "uninfected")]:
                if 'spatial' not in adata.obsm:
                    raise ValueError(f"No spatial coordinates found in {label} dataset")
                logger.info(f"Verified spatial coordinates for {label} dataset")
        
            # Ensure raw counts are used
            if 'raw_counts' in self.sc_adata.layers:
                self.sc_adata.X = self.sc_adata.layers['raw_counts'].copy()
        
            # Make gene names unique
            self.sc_adata.var_names_make_unique()
            self.visium_infected.var_names_make_unique()
            self.visium_uninfected.var_names_make_unique()
        
            # Get common genes
            common_genes = list(set(self.visium_infected.var_names) &
                                set(self.visium_uninfected.var_names) &
                                set(self.sc_adata.var_names))
        
            if len(common_genes) == 0:
                raise ValueError("No common genes found between datasets")
        
            logger.info(f"Found {len(common_genes)} common genes across datasets")
        
            # Subset to common genes
            self.visium_infected = self.visium_infected[:, common_genes].copy()
            self.visium_uninfected = self.visium_uninfected[:, common_genes].copy()
            self.sc_adata = self.sc_adata[:, common_genes].copy()
        
            # Setup spatial neighbors
            logger.info("Setting up spatial neighbors...")
            for visium, label in [(self.visium_infected, "infected"),
                                (self.visium_uninfected, "uninfected")]:
                sq.gr.spatial_neighbors(visium)
            
        except Exception as e:
            logger.error(f"Data setup failed: {str(e)}")
            raise

    def run_deconvolution(self, n_epochs=1000, batch_size=2500, gpu=True):
        """Run cell2location deconvolution"""
        logger.info("Running cell type deconvolution...")
    
        try:
            if gpu and not torch.cuda.is_available():
                logger.warning("GPU requested but not available. Falling back to CPU.")
                gpu = False
        
            accelerator = "gpu" if gpu else "cpu"
        
            # Define batch mapping
            batch_mapping = {
                'SLN111-D1_SingleCell_Totalvi_111_uninfected': 'SLN111_uninfected',
                'SLN111-D2_SingleCell_Totalvi_111_uninfected': 'SLN111_uninfected',
                'SLN208-D1_SingleCell_Totalvi_208_uninfected': 'SLN208_uninfected',
                'SLN208-D2_SingleCell_Totalvi_208_uninfected': 'SLN208_uninfected',
                'Spleen_SingleCell_Exp1A_uninfected': 'Exp1A_uninfected',
                'Spleen_SingleCell_Exp1A_3wk_infected': 'Exp1A_infected',
                'Spleen_SingleCell_Exp2A_uninfected': 'Exp2A_uninfected',
                'Spleen_SingleCell_Exp2A_3wk_infected': 'Exp2A_infected',
                'Spleen_SingleCell_Exp3A_3wk_infected_1': 'Exp3A_infected',
                'Spleen_SingleCell_Exp3A_3wk_infected_2': 'Exp3A_infected',
                'Spleen_SingleCell_Exp5A_uninfected': 'Exp5A_uninfected',
                'Spleen_SingleCell_Exp5A_3wk_infected': 'Exp5A_infected'
            }
        
            self.sc_adata.obs['batch'] = self.sc_adata.obs['dataset'].map(batch_mapping)
            self.sc_adata.obs['batch'] = pd.Categorical(self.sc_adata.obs['batch'])
        
            # Setup reference model
            cell2location.models.RegressionModel.setup_anndata(
                self.sc_adata,
                batch_key='batch',
                labels_key='celltypes',  # Uses the updated annotations
                categorical_covariate_keys=['health', 'experiment']
            )
        
            ref_model = cell2location.models.RegressionModel(self.sc_adata)
            ref_model.train(
                max_epochs=n_epochs,
                accelerator=accelerator,
                progress_bar_refresh_rate=1
            )
        
            # Export posterior
            self.sc_adata = ref_model.export_posterior(
                self.sc_adata,
                sample_kwargs={'num_samples': 1000, 'batch_size': batch_size}
            )
        
            # Extract cluster-specific expression
            if 'means_per_cluster_mu_fg' in self.sc_adata.varm.keys():
                inf_aver = self.sc_adata.varm['means_per_cluster_mu_fg'][[
                    f'means_per_cluster_mu_fg_{i}' for i in self.sc_adata.uns['mod']['factor_names']
                ]].copy()
            else:
                inf_aver = self.sc_adata.var[[
                    f'means_per_cluster_mu_fg_{i}' for i in self.sc_adata.uns['mod']['factor_names']
                ]].copy()
        
            inf_aver.columns = self.sc_adata.uns['mod']['factor_names']
            ref_summary = inf_aver
        
            # Process spatial data
            for condition, visium_data in tqdm([
                ('infected', self.visium_infected),
                ('uninfected', self.visium_uninfected)
            ], desc="Processing conditions"):
            
                cell2location.models.Cell2location.setup_anndata(visium_data)
            
                model = cell2location.models.Cell2location(
                    visium_data,
                    cell_state_df=ref_summary,
                    N_cells_per_location=30,
                    detection_alpha=20
                )
            
                model.train(
                    max_epochs=n_epochs,
                    accelerator=accelerator,
                    progress_bar_refresh_rate=1
                )
            
                # Export results
                visium_data = model.export_posterior(
                    visium_data,
                    sample_kwargs={'num_samples': 1000, 'batch_size': model.adata.n_obs}
                )
            
                # Store results
                if condition == 'infected':
                    self.visium_infected = visium_data
                else:
                    self.visium_uninfected = visium_data
                
            return True
        
        except Exception as e:
            logger.error(f"Deconvolution failed: {str(e)}")
            if gpu:
                torch.cuda.empty_cache()
            raise

    def visualize_celltypes(self, output_dir="./celltype_plots", max_cells_per_plot=1):
        """Visualize spatial distribution of cell types using cell2location plotting"""
        logger.info("Visualizing cell type spatial distributions...")
    
        try:
            from cell2location.plt import plot_spatial
            output_dir = Path(output_dir)
            output_dir.mkdir(exist_ok=True, parents=True)
        
            for condition, adata in [
                ('Infected', self.visium_infected),
                ('Uninfected', self.visium_uninfected)
            ]:
                logger.info(f"Processing {condition} dataset...")
            
                # Get cell type names from the model
                if 'mod' not in adata.uns:
                    logger.error(f"No model results found in {condition} dataset")
                    continue
            
                # Use the updated cell types from the model
                cell_types = self.sc_adata.obs['celltypes'].unique()
                logger.info(f"Found {len(cell_types)} cell types")
            
                # Add cell abundance data to obs
                if 'q05_cell_abundance_w_sf' not in adata.obsm:
                    logger.error(f"Cell abundance data not found in {condition} dataset")
                    continue


                
                adata.obs[cell_types] = adata.obsm['q05_cell_abundance_w_sf']
            
                # Process cell types in groups of max_cells_per_plot
                for i in range(0, len(cell_types), max_cells_per_plot):
                    group = cell_types[i:i + max_cells_per_plot]
                
                    try:
                        with plt.rc_context({'figure.figsize': (15, 15)}):
                            fig = plot_spatial(
                                adata=adata,
                                color=group,  # cell types to plot
                                labels=group,  # labels to show
                                show_img=True,
                                style='fast',
                                max_color_quantile=0.992,
                                circle_diameter=6,
                                colorbar_position='right'
                            )
                        
                            # Save figure
                            group_name = f"celltypes_{i//max_cells_per_plot}"
                            output_file = output_dir / f"{condition}_{group_name}.png"
                            plt.savefig(
                                output_file,
                                dpi=300,
                                bbox_inches='tight',
                                facecolor='white'
                            )
                            plt.close()
                        
                    except Exception as e:
                        logger.error(f"Error plotting group {i//max_cells_per_plot} in {condition}: {str(e)}")
                        plt.close()
                        continue
            
                # Clean up added obs columns
                adata.obs = adata.obs.drop(columns=cell_types)
                    
            return True
        
        except Exception as e:
            logger.error(f"Cell type visualization failed: {str(e)}")
            plt.close('all')
            raise
            
    def calculate_morans_i(self, adata, values, coords):
        """Calculate Moran's I statistic for spatial autocorrelation"""
        try:
            # Create weights matrix
            kdt = libpysal.weights.KNN.from_array(coords, k=8)
            kdt.transform = 'R'  # Row-standardize weights
            
            # Calculate Moran's I
            moran = Moran(values, kdt)
            
            return {
                'I': moran.I,
                'p_value': moran.p_sim,
                'z_score': moran.z_sim
            }
        except Exception as e:
            logger.error(f"Moran's I calculation failed: {str(e)}")
            return None

    def detect_hotspots(self, values, coords, threshold=1.5):
        """Detect spatial hotspots using local statistics"""
        try:
            # Convert Series to numpy array if needed
            if isinstance(values, pd.Series):
                values = values.to_numpy()
            
            # Standardize values
            scaler = StandardScaler()
            values_std = scaler.fit_transform(values.reshape(-1, 1)).flatten()
        
            # Create spatial weights matrix
            kdt = libpysal.weights.KNN.from_array(coords, k=8)
            kdt.transform = 'R'
        
            # Calculate local Moran's I
            local_moran = esda.moran.Moran_Local(values_std, kdt)
        
            # Identify significant clusters
            sig = 0.05
            hotspots = (local_moran.p_sim < sig) & (values_std > threshold)
            coldspots = (local_moran.p_sim < sig) & (values_std < -threshold)
        
            return {
                'hotspots': hotspots,
                'coldspots': coldspots,
                'p_values': local_moran.p_sim,
                'local_I': local_moran.Is
            }
        except Exception as e:
            logger.error(f"Hotspot detection failed: {str(e)}")
            return None

    
    def calculate_interaction_scores(self, adata, sender, receiver, ligand, receptor):
        """Calculate cell-type specific interaction scores"""
        try:
            # Add prefix to match cell type names in the abundance matrix
            sender_key = f"q05cell_abundance_w_sf_{sender}"
            receiver_key = f"q05cell_abundance_w_sf_{receiver}"
        
            # Get cell type abundances
            sender_abundance = adata.obsm['q05_cell_abundance_w_sf'][sender_key]
            receiver_abundance = adata.obsm['q05_cell_abundance_w_sf'][receiver_key]
        
            # Debug info
            logger.debug(f"Looking for ligand: {ligand}")
            logger.debug(f"Looking for receptor: {receptor}")
            logger.debug(f"Available genes (first 10): {list(adata.var_names[:10])}")
        
            # Handle ligand - case insensitive matching
            ligand_clean = ligand.upper()
            var_names_upper = [g.upper() for g in adata.var_names]
            if ligand_clean not in var_names_upper:
                raise ValueError(f"Ligand gene '{ligand}' not found in dataset")
            ligand_idx = var_names_upper.index(ligand_clean)
            ligand_expr = adata[:, adata.var_names[ligand_idx]].X.toarray().flatten()
        
            # Handle receptor - might be composite (e.g., CD74_CD44)
            receptor_parts = receptor.split('_')
            receptor_expr = np.ones(adata.n_obs)
        
            for rec_part in receptor_parts:
                rec_clean = rec_part.upper()
                if rec_clean not in var_names_upper:
                    raise ValueError(f"Receptor part '{rec_part}' not found in dataset")
                rec_idx = var_names_upper.index(rec_clean)
                receptor_expr *= adata[:, adata.var_names[rec_idx]].X.toarray().flatten()
        
            # Calculate interaction score
            interaction_score = sender_abundance * receiver_abundance * ligand_expr * receptor_expr
        
            # Calculate spatial statistics
            coords = adata.obsm['spatial']
            morans_stats = self.calculate_morans_i(adata, interaction_score, coords)
            hotspot_stats = self.detect_hotspots(interaction_score, coords)
        
            return {
                'interaction_score': interaction_score,
                'morans_stats': morans_stats,
                'hotspot_stats': hotspot_stats,
                'sender_abundance': sender_abundance,
                'receiver_abundance': receiver_abundance,
                'ligand_expression': ligand_expr,
                'receptor_expression': receptor_expr
            }
        except Exception as e:
            logger.error(f"Interaction score calculation failed for {sender}-{receiver}: {str(e)}")
            return None



    def compare_conditions(self, infected_scores, uninfected_scores):
        """Compare interaction patterns between conditions"""
        try:
            # Check if either score calculation failed
            if infected_scores is None or uninfected_scores is None:
                raise ValueError("One or both score calculations failed")
           
            # Perform statistical tests
            stats_test = stats.mannwhitneyu(
                infected_scores['interaction_score'],
                uninfected_scores['interaction_score'],
                alternative='two-sided'
            )
       
            # Calculate fold change
            mean_infected = np.mean(infected_scores['interaction_score'])
            mean_uninfected = np.mean(uninfected_scores['interaction_score'])
            fold_change = mean_infected / mean_uninfected if mean_uninfected != 0 else float('inf')
       
            # Compare spatial patterns
            spatial_difference = {
                'infected_morans': infected_scores['morans_stats'],
                'uninfected_morans': uninfected_scores['morans_stats'],
                'infected_hotspots': infected_scores['hotspot_stats'],
                'uninfected_hotspots': uninfected_scores['hotspot_stats']
            }
       
            return {
                'statistical_test': stats_test,
                'fold_change': fold_change,
                'spatial_difference': spatial_difference
            }
        except Exception as e:
            logger.error(f"Condition comparison failed: {str(e)}")
            return None
        
    def visualize_lr_patterns(self, output_dir="./lr_plots", dpi=300, fold_change_threshold=2.0, figsize=(20, 25)):
        """Visualize spatial patterns of L-R interactions"""
        logger.info("Visualizing L-R spatial patterns...")
        
        output_dir = Path(output_dir)
        output_dir.mkdir(exist_ok=True, parents=True)
        
        plt.ioff()
        
        # Debug information
        logger.info("First few L-R interactions:")
        logger.info(self.lr_interactions.head())
        logger.info(f"Total interactions: {len(self.lr_interactions)}")
        
        # Filter for highly increased interactions
        increased_interactions = self.lr_interactions[
            (self.lr_interactions['Fold_Change'] >= fold_change_threshold) &
            (self.lr_interactions['Direction'] == 'up')
        ]
        
        logger.info(f"Found {len(increased_interactions)} interactions with fold change >= {fold_change_threshold}")
        
        for _, interaction in tqdm(increased_interactions.iterrows(), 
                                desc="Processing increased interactions"):
            try:
                sender = interaction['Sender']
                receiver = interaction['Receiver']
                ligand = interaction['ligand']
                receptor = interaction['receptor']
                
                # Calculate interaction scores for both conditions
                infected_scores = self.calculate_interaction_scores(
                    self.visium_infected, sender, receiver, ligand, receptor
                )
                uninfected_scores = self.calculate_interaction_scores(
                    self.visium_uninfected, sender, receiver, ligand, receptor
                )
                
                if infected_scores is None or uninfected_scores is None:
                    continue
                    
                # Compare conditions
                comparison = self.compare_conditions(infected_scores, uninfected_scores)
                
                if comparison is None:
                    continue
                
                # Create figure with high resolution
                fig = plt.figure(figsize=figsize)
                gs = plt.GridSpec(5, 3)
                
                for idx, (condition, scores, strength) in enumerate([
                    ('Infected (3wk)', infected_scores, interaction['Strength_3wk']),
                    ('Uninfected (0wk)', uninfected_scores, interaction['Strength_0wk'])
                ]):
                    base_idx = idx * 2
                    
                    # Plot sender cells
                    ax = fig.add_subplot(gs[base_idx, 0])
                    sc.pl.spatial(
                        self.visium_infected if idx == 0 else self.visium_uninfected,
                        color=sender,
                        img_key='hires',
                        title=f"{condition} - {sender} Abundance",
                        color_map='viridis',
                        spot_size=100,
                        show=False,
                        ax=ax
                    )
                    
                    # Plot receiver cells
                    ax = fig.add_subplot(gs[base_idx, 1])
                    sc.pl.spatial(
                        self.visium_infected if idx == 0 else self.visium_uninfected,
                        color=receiver,
                        img_key='hires',
                        title=f"{condition} - {receiver} Abundance",
                        color_map='viridis',
                        spot_size=100,
                        show=False,
                        ax=ax
                    )
                    
                    # Plot interaction strength
                    ax = fig.add_subplot(gs[base_idx, 2])
                    adata = self.visium_infected if idx == 0 else self.visium_uninfected
                    adata.obs['interaction_strength'] = scores['interaction_score']
                    sc.pl.spatial(
                        adata,
                        color='interaction_strength',
                        img_key='hires',
                        title=f"{condition} - Interaction Strength\n(Strength = {strength:.4f})",
                        color_map='viridis',
                        spot_size=100,
                        show=False,
                        ax=ax
                    )
                    
                    # Plot hotspots
                    ax = fig.add_subplot(gs[base_idx + 1, :])
                    self.plot_hotspots(
                        adata,
                        scores['hotspot_stats'],
                        title=f"{condition} - Interaction Hotspots",
                        ax=ax
                    )
                
                # Add statistical summary
                ax = fig.add_subplot(gs[4, :])
                self.plot_statistical_summary(
                    interaction,
                    comparison,
                    infected_scores,
                    uninfected_scores,
                    ax=ax
                )
                
                # Add overall title
                plt.suptitle(
                    f"Spatial Analysis of {sender}-{receiver} Interaction\n"
                    f"Ligand-Receptor: {ligand}-{receptor}\n"
                    f"Fold Change: {interaction['Fold_Change']:.3f} ↑\n"
                    f"Direction: {interaction['Direction']}",
                    fontsize=14, y=1.02
                )
                
                plt.tight_layout()
                
                # Save figure with high resolution
                safe_filename = f"{sender}_{receiver}_{ligand}_{receptor}_spatial_pattern".replace('/', '-')
                fig.savefig(
                    output_dir / f"{safe_filename}.png",
                    dpi=dpi,
                    bbox_inches='tight',
                    facecolor='white'
                )
                plt.close(fig)
                
            except Exception as e:
                logger.error(f"Error processing {sender}-{receiver} interaction: {str(e)}")
                plt.close('all')
                continue
        
        plt.ion()
        logger.info("Visualization complete")

    def plot_hotspots(self, adata, hotspot_stats, title, ax):
        """Plot hotspots and coldspots"""
        try:
            # Create categorical colors for spots
            colors = np.full(len(adata), 'lightgrey')
            colors[hotspot_stats['hotspots']] = 'red'
            colors[hotspot_stats['coldspots']] = 'blue'
            
            # Plot spots
            ax.scatter(
                adata.obsm['spatial'][:, 0],
                adata.obsm['spatial'][:, 1],
                c=colors,
                alpha=0.7,
                s=50
            )
            
            # Add tissue image if available
            if 'hires' in adata.uns['spatial']:
                img = adata.uns['spatial']['hires']
                ax.imshow(
                    img,
                    alpha=0.3,
                    extent=adata.uns['spatial']['hires_scalef']
                )
            
            ax.set_title(title)
            ax.axis('equal')
            
        except Exception as e:
            logger.error(f"Hotspot plotting failed: {str(e)}")

    def plot_statistical_summary(self, interaction, comparison, infected_scores, 
                                uninfected_scores, ax):
        """Plot statistical summary of interaction comparison"""
        try:
            # Create summary text
            summary_text = (
                f"Statistical Summary:\n"
                f"Mann-Whitney U test p-value: {comparison['statistical_test'].pvalue:.2e}\n"
                f"Fold Change: {comparison['fold_change']:.2f}\n\n"
                f"Spatial Statistics:\n"
                f"Infected Moran's I: {infected_scores['morans_stats']['I']:.3f} "
                f"(p={infected_scores['morans_stats']['p_value']:.2e})\n"
                f"Uninfected Moran's I: {uninfected_scores['morans_stats']['I']:.3f} "
                f"(p={uninfected_scores['morans_stats']['p_value']:.2e})\n"
            )
            
            ax.text(
                0.5, 0.5,
                summary_text,
                ha='center',
                va='center',
                transform=ax.transAxes,
                bbox=dict(facecolor='white', alpha=0.8, edgecolor='none')
            )
            ax.axis('off')
            
        except Exception as e:
            logger.error(f"Statistical summary plotting failed: {str(e)}")

In [4]:
def main():
    """Main function to run the analysis"""
    try:
        # Set paths using your directory structure
        paths = {
            'visium_infected': '/home/robeylab/Desktop/outs_infected',
            'visium_uninfected': '/home/robeylab/Desktop/outs_UNINFECTED',
            'lr_interactions': '/home/robeylab/Desktop/cellchat_sorted.csv',
            'output': './results4'
        }
       
        # Load single-cell data (old data with raw_counts)
        logger.info("Loading original single-cell reference data...")
        sc_data = sc.read_h5ad('/home/robeylab/cellxgene_data/preprocessed_deconvolution_sc_spleen_240116.h5ad')
       
        # Load new annotations data
        logger.info("Loading updated cell type annotations...")
        new_annotations = sc.read_h5ad('/home/robeylab/cellxgene_data/processed_deconvolution_sc_spleen_240713_fixed_full_data.h5ad')
       
        # Initialize analyzer with both AnnData objects
        logger.info("Initializing spatial L-R analyzer...")
        analyzer = SpatialLRAnalysis(
            visium_path_infected=paths['visium_infected'],
            visium_path_uninfected=paths['visium_uninfected'],
            sc_adata=sc_data,
            new_annotations_adata=new_annotations,
            lr_interactions_path=paths['lr_interactions'],
            output_dir=paths['output']
        )
       
        # Run deconvolution
        logger.info("Starting deconvolution analysis...")
        analyzer.run_deconvolution(n_epochs=1000, batch_size=2500, gpu=True)
       
        # Create output directories
        output_dirs = {
            'lr_plots': Path(paths['output']) / 'lr_plots',
            'statistics': Path(paths['output']) / 'statistics',
            'hotspots': Path(paths['output']) / 'hotspots',
            'celltype_plots': Path(paths['output']) / 'celltype_plots'
        }
       
        for dir_path in output_dirs.values():
            dir_path.mkdir(parents=True, exist_ok=True)
       
        # Visualize cell types
        logger.info("Generating cell type visualizations...")
        analyzer.visualize_celltypes(output_dir=output_dirs['celltype_plots'])

        # Visualize highly increased L-R interactions
        logger.info("Generating L-R interaction visualizations...")
        analyzer.visualize_lr_patterns(output_dir=output_dirs['lr_plots'], fold_change_threshold=2.0)
       
        # Save summary statistics
        logger.info("Saving summary statistics...")
        summary_stats = []
       
        for _, interaction in analyzer.lr_interactions.iterrows():
            try:
                sender = interaction['Sender']
                receiver = interaction['Receiver']
                ligand = interaction['ligand']
                receptor = interaction['receptor']
               
                # Calculate scores
                infected_scores = analyzer.calculate_interaction_scores(
                    analyzer.visium_infected, sender, receiver, ligand, receptor
                )
                uninfected_scores = analyzer.calculate_interaction_scores(
                    analyzer.visium_uninfected, sender, receiver, ligand, receptor
                )
               
                # Compare conditions
                comparison = analyzer.compare_conditions(infected_scores, uninfected_scores)
               
                # Compile statistics
                stats_dict = {
                    'Sender': sender,
                    'Receiver': receiver,
                    'Ligand': ligand,
                    'Receptor': receptor,
                    'Fold_Change': comparison['fold_change'],
                    'P_Value': comparison['statistical_test'].pvalue,
                    'Infected_Morans_I': infected_scores['morans_stats']['I'],
                    'Infected_Morans_P': infected_scores['morans_stats']['p_value'],
                    'Uninfected_Morans_I': uninfected_scores['morans_stats']['I'],
                    'Uninfected_Morans_P': uninfected_scores['morans_stats']['p_value'],
                    'Infected_Hotspots': np.sum(infected_scores['hotspot_stats']['hotspots']),
                    'Uninfected_Hotspots': np.sum(uninfected_scores['hotspot_stats']['hotspots'])
                }
               
                summary_stats.append(stats_dict)
               
            except Exception as e:
                logger.error(f"Error processing {sender}-{receiver} interaction: {str(e)}")
                continue
       
        # Save summary statistics
        summary_df = pd.DataFrame(summary_stats)
        summary_df.to_csv(output_dirs['statistics'] / 'interaction_summary_stats.csv', index=False)
       
        logger.info("Analysis complete!")
       
    except Exception as e:
        logger.error(f"Analysis failed: {str(e)}")
        raise
   
    finally:
        plt.close('all')
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

if __name__ == "__main__":
    main()

2024-12-11 15:28:12 - INFO - Loading original single-cell reference data...
2024-12-11 15:28:13 - INFO - Loading updated cell type annotations...
2024-12-11 15:28:13 - INFO - Initializing spatial L-R analyzer...
2024-12-11 15:28:13 - INFO - Initializing analysis...
2024-12-11 15:28:13 - INFO - Loading Visium data...
2024-12-11 15:28:14 - INFO - Updating cell type annotations...
2024-12-11 15:28:14 - ERROR - Failed to update cell type annotations: "['ATAGGCTTCTCTTCAA-1_Spleen_SingleCell_Exp2A_3wk_infected', 'ATTCTTGTCGTCAGAT-1_Spleen_SingleCell_Exp2A_3wk_infected', 'CAAGGGACAACGACAG-1_Spleen_SingleCell_Exp2A_3wk_infected', 'CACGAATAGTCAACAA-1_Spleen_SingleCell_Exp2A_3wk_infected', 'CTTCGGTAGCAAGTCG-1_Spleen_SingleCell_Exp2A_3wk_infected', 'GACATCAAGACAGTCG-1_Spleen_SingleCell_Exp2A_3wk_infected', 'GACGTTAGTATCGTAC-1_Spleen_SingleCell_Exp2A_3wk_infected', 'GCTGCAGAGTAAGCAT-1_Spleen_SingleCell_Exp2A_3wk_infected', 'GGGCGTTGTGTTACTG-1_Spleen_SingleCell_Exp2A_3wk_infected', 'GGGTTTACAGTCTCT

KeyError: "['ATAGGCTTCTCTTCAA-1_Spleen_SingleCell_Exp2A_3wk_infected', 'ATTCTTGTCGTCAGAT-1_Spleen_SingleCell_Exp2A_3wk_infected', 'CAAGGGACAACGACAG-1_Spleen_SingleCell_Exp2A_3wk_infected', 'CACGAATAGTCAACAA-1_Spleen_SingleCell_Exp2A_3wk_infected', 'CTTCGGTAGCAAGTCG-1_Spleen_SingleCell_Exp2A_3wk_infected', 'GACATCAAGACAGTCG-1_Spleen_SingleCell_Exp2A_3wk_infected', 'GACGTTAGTATCGTAC-1_Spleen_SingleCell_Exp2A_3wk_infected', 'GCTGCAGAGTAAGCAT-1_Spleen_SingleCell_Exp2A_3wk_infected', 'GGGCGTTGTGTTACTG-1_Spleen_SingleCell_Exp2A_3wk_infected', 'GGGTTTACAGTCTCTC-1_Spleen_SingleCell_Exp2A_3wk_infected', 'GTCCACTCATACCATG-1_Spleen_SingleCell_Exp2A_3wk_infected', 'TACACCCCACCTGCTT-1_Spleen_SingleCell_Exp2A_3wk_infected', 'TACGTCCCATGTTACG-1_Spleen_SingleCell_Exp2A_3wk_infected', 'TACTTACTCTGGGCGT-1_Spleen_SingleCell_Exp2A_3wk_infected', 'TATCCTAGTCGCATCG-1_Spleen_SingleCell_Exp2A_3wk_infected', 'TCATTGTCAATCTAGC-1_Spleen_SingleCell_Exp2A_3wk_infected', 'TCGGGCAAGAGATTCA-1_Spleen_SingleCell_Exp2A_

In [1]:
import pandas as pd

# Load the Excel files
file1 = pd.read_excel('/Users/yashkulkarni/Downloads/IFN1_signaling_pathway.xlsx')
file2 = pd.read_excel('/Users/yashkulkarni/Downloads/IL27_signaling_pathway.xlsx')

# Extract the 'Symbol' column
symbols1 = set(file1['Symbol'])
symbols2 = set(file2['Symbol'])

# Find common and unique symbols
common_symbols = symbols1.intersection(symbols2)
unique_to_file1 = symbols1 - symbols2
unique_to_file2 = symbols2 - symbols1

# Print the results
print("Common Symbols:", common_symbols)
print("Unique to File 1:", unique_to_file1)
print("Unique to File 2:", unique_to_file2)

Common Symbols: {'Oas1a', 'Oas1g', 'Oas1c', 'Oas1d', 'Oasl1', 'Oas1e', 'Oas3', 'Oas2', 'Stat1', 'Oas1h', 'Oas1f', 'Oas1b', 'Oasl2'}
Unique to File 1: {'Gm13275', 'Mmp12', 'Stat2', 'Nlrc5', 'Ifitm6', 'Ifna6', 'Ythdf3', 'Ikbke', 'Ifna5', 'Ifne', 'Ifna7', 'Ifitm2', 'Trex1', 'Rnf185', 'Adar', 'Cactin', 'Irf7', 'Wnt5a', 'Cnot7', 'Usp27x', 'Ifitm7', 'Ifna9', 'Rbm47', 'Ifitm1', 'Jak1', 'Tbk1', 'Ifnab', 'Ifna11', 'Gm13272', 'Gpr108', 'Irf3', 'Ifna16', 'Usp18', 'Ifna2', 'Ifna14', 'Mul1', 'Cdc37', 'Tyk2', 'Ifna4', 'Ube2k', 'Ifih1', 'Myd88', 'Gm13276', 'Mavs', 'Ttll12', 'Sting1', 'Isg15', 'Ifna13', 'Gm13277', 'Smim30', 'Ifna1', 'Gm13283', 'Ifitm3', 'Ifnar2', 'Ifnz', 'Trim56', 'Ythdf2', 'Trim6', 'Hdac4', 'Mettl3', 'Dcst1', 'Eif4e2', 'Irak1', 'Ifnk', 'Samhd1', 'Ifna12', 'Trim41', 'Gigyf2', 'Sin3a', 'Lsm14a', 'Ptpn2', 'Ifnar1', 'Fadd', 'Gm13271', 'Ifna15', 'Usp29', 'Zbp1', 'Ifnb1', 'Trim65'}
Unique to File 2: {'Il27ra', 'Il6st'}


/opt/anaconda3/lib/python3.11/site-packages/openpyxl/styles/stylesheet.py:226: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


In [1]:
import pandas as pd

def compare_gene_lists(ifn_path, il27_path):
    # Read Excel files
    ifn_df = pd.read_excel(ifn_path)
    il27_df = pd.read_excel(il27_path)
    
    # Get gene lists (assuming the gene column is the first column - adjust if needed)
    ifn_genes = set(ifn_df.iloc[:, 0])
    il27_genes = set(il27_df.iloc[:, 0])
    
    # Find common and unique genes
    common_genes = ifn_genes.intersection(il27_genes)
    ifn_unique = ifn_genes.difference(il27_genes)
    il27_unique = il27_genes.difference(ifn_genes)
    
    # Create DataFrame for results
    results = pd.DataFrame({
        'Common_Genes': list(common_genes) + [''] * (max(len(ifn_unique), len(il27_unique)) - len(common_genes)),
        'IFN_Unique': list(ifn_unique) + [''] * (max(len(common_genes), len(il27_unique)) - len(ifn_unique)),
        'IL27_Unique': list(il27_unique) + [''] * (max(len(common_genes), len(ifn_unique)) - len(il27_unique))
    })
    
    # Save results
    output_path = "/Users/yashkulkarni/Downloads/gene_comparison_results.xlsx"
    results.to_excel(output_path, index=False)
    
    # Print summary
    print(f"Found {len(common_genes)} common genes")
    print(f"Found {len(ifn_unique)} genes unique to IFN")
    print(f"Found {len(il27_unique)} genes unique to IL27")
    print(f"\nResults saved to: {output_path}")
    
    return results

# File paths
ifn_path = "/Users/yashkulkarni/Downloads/IFN1_dictionary.xlsx"
il27_path = "/Users/yashkulkarni/Downloads/IL27_dictionary.xlsx"

# Run comparison
results = compare_gene_lists(ifn_path, il27_path)

# Display first few rows of each category
print("\nFirst few common genes:")
print(results['Common_Genes'].head())
print("\nFirst few IFN-unique genes:")
print(results['IFN_Unique'].head())
print("\nFirst few IL27-unique genes:")
print(results['IL27_Unique'].head())

Found 209 common genes
Found 533 genes unique to IFN
Found 332 genes unique to IL27

Results saved to: /Users/yashkulkarni/Downloads/gene_comparison_results.xlsx

First few common genes:
0       Bst2
1       Gbp7
2     H2-T22
3      Ifih1
4    Trim30a
Name: Common_Genes, dtype: object

First few IFN-unique genes:
0       Map2k1
1         Tlr3
2    D17Wsu92e
3        Ywhaz
4        Art2b
Name: IFN_Unique, dtype: object

First few IL27-unique genes:
0             Wbp1
1    A230072E10Rik
2             Ccr5
3           Dusp12
4            Mrps5
Name: IL27_Unique, dtype: object
